# Notebook 01 — Celeb-DF v2 Extraction and Actor-Disjoint Sampling (v2)

**Goal:** Build train/val/test splits over Celeb-DF v2 where:
1. **Test set = Celeb-DF's official `List_of_testing_videos.txt`** — the canonical test split used in every Celeb-DF paper (AVTENet, EfficientNet-V2S, etc.). Using it makes our results directly comparable to published baselines.
2. **Train/Val splits are actor-disjoint by SOURCE identity** — for fake clips `idN_idM_XXXX.mp4`, we hash idN (the face donor). idM (the body/audio donor) is allowed to span splits since it's not the discriminative signal for face manipulation detection. This is the protocol used by Li et al. 2020 and most Celeb-DF follow-ups.
3. **Deterministic, reproducible** — seed = 42.

**Why this differs from v1:** v1 required BOTH source AND target identities to land in the same split for synthesis clips. With 80/10/10 hashing, P(both → val) = 0.01, P(both → test) = 0.01 — which collapsed val and test sets to <100 clips each and inverted class balance. v2 fixes this by (a) using the official test list (no hashing for test) and (b) hashing only by source identity for train/val.

**Outputs:**
- `data/celeb-df/...` — extracted videos
- `data/manifest.csv` — full clip-level manifest
- `data/splits/{train,val,test}.csv` — per-split manifests
- `data/official_test_list.csv` — parsed version of `List_of_testing_videos.txt` for reference

## 1. Imports and configuration

In [ ]:
import os
import sys
import re
import json
import shutil
import hashlib
import zipfile
import shutil as _shutil
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from tqdm.auto import tqdm

print(f"Python: {sys.version.split()[0]}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path('..').resolve()

# Paths
DATA_DIR          = PROJECT_ROOT / 'data'
ZIP_PATH          = DATA_DIR / 'raw_zips' / 'Celeb-DF-v2.zip'
CELEB_DIR         = DATA_DIR / 'celeb-df'
SPLITS_DIR        = DATA_DIR / 'splits'
MANIFEST_CSV      = DATA_DIR / 'manifest.csv'
TEST_LIST_PATH    = CELEB_DIR / 'List_of_testing_videos.txt'
OFFICIAL_TEST_CSV = DATA_DIR / 'official_test_list.csv'

# For train/val split (test is fixed by official list)
TRAIN_VAL_RATIOS = {'train': 0.90, 'val': 0.10}

# Reproducibility seed
SAMPLE_SEED = 42

# Delete zip after successful extraction
DELETE_ZIP_AFTER_EXTRACT = True

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Zip path:     {ZIP_PATH}")
print(f"Extract to:   {CELEB_DIR}")
print(f"Test list:    {TEST_LIST_PATH}")
print(f"Train/Val:    {TRAIN_VAL_RATIOS}")

## 2. Folder setup

In [ ]:
for d in [DATA_DIR, CELEB_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  OK: {d}")
print("\nFolder structure ready.")

## 3. Extract Celeb-DF v2 zip (idempotent — skips if already extracted)

If you already extracted on a previous run, this cell will detect the files on disk and skip re-extraction.

In [ ]:
def get_free_gb(path: Path) -> float:
    return _shutil.disk_usage(path).free / (1024**3)


def extract_zip_to(zip_path: Path, output_dir: Path) -> dict:
    counts = Counter()
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if m.endswith('.mp4')]
        meta_members = [m for m in zf.namelist() if m.endswith('.txt') and not m.endswith('/')]
        
        for m in meta_members:
            dst = output_dir / m
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                with zf.open(m) as src, open(dst, 'wb') as out:
                    shutil.copyfileobj(src, out)
                print(f"  Extracted metadata: {m}")
        
        for member in tqdm(members, desc="Extracting videos", unit="clip"):
            dst = output_dir / member
            if dst.exists():
                counts[member.split('/')[0]] += 1
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(dst, 'wb') as out:
                shutil.copyfileobj(src, out)
            counts[member.split('/')[0]] += 1
    return dict(counts)


# Skip if already fully extracted
existing = list(CELEB_DIR.rglob('*.mp4'))
if len(existing) >= 6000:
    print(f"[SKIP] Already have {len(existing):,} videos in {CELEB_DIR}. Re-extraction not needed.")
    extracted_counts = Counter(p.parts[-2] for p in existing)
    print("Found on disk:")
    for folder, count in sorted(extracted_counts.items()):
        print(f"  {folder}: {count:,} videos")
elif not ZIP_PATH.exists():
    print(f"[ERROR] Zip not found at {ZIP_PATH}")
    print("Did you move Celeb-DF-v2.zip to data/raw_zips/?")
    raise FileNotFoundError(str(ZIP_PATH))
else:
    print(f"Free disk before extraction: {get_free_gb(DATA_DIR):.1f} GB\n")
    extracted_counts = extract_zip_to(ZIP_PATH, CELEB_DIR)
    print(f"\nExtraction complete:")
    for folder, count in sorted(extracted_counts.items()):
        print(f"  {folder}: {count:,} videos")
    print(f"\nFree disk after extraction: {get_free_gb(DATA_DIR):.1f} GB")

In [ ]:
# Delete zip after successful extraction (saves ~9 GB)
if DELETE_ZIP_AFTER_EXTRACT and ZIP_PATH.exists():
    total_on_disk = len(list(CELEB_DIR.rglob('*.mp4')))
    if total_on_disk >= 6000:
        ZIP_PATH.unlink()
        print(f"[OK] Deleted zip. Free disk now: {get_free_gb(DATA_DIR):.1f} GB")
    else:
        print(f"[WARN] Only {total_on_disk} videos extracted. Keeping zip as safety net.")
elif not ZIP_PATH.exists():
    print("Zip already deleted (probably from a previous successful run).")
else:
    print("Keeping zip (DELETE_ZIP_AFTER_EXTRACT = False).")

## 4. Parse Celeb-DF's official test set list

`List_of_testing_videos.txt` defines the canonical test split. Each line follows the format:
```
<label> <path>
```
where label is `1` (real) or `0` (fake), and path is relative like `Celeb-real/id14_0001.mp4`.

This is the test set used by every published Celeb-DF paper — our results will be directly comparable to AVTENet, AVT+, and other AV-XAI baselines.

In [ ]:
if not TEST_LIST_PATH.exists():
    raise FileNotFoundError(
        f"{TEST_LIST_PATH} not found. The zip should have included it — "
        "maybe it didn't extract? Check the celeb-df folder."
    )

test_records = []
with open(TEST_LIST_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 2:
            continue
        official_label = parts[0]  # '1' or '0'
        rel_path = parts[1].replace('\\', '/')  # normalize separators
        test_records.append({'official_label': official_label, 'rel_path': rel_path})

official_test_df = pd.DataFrame(test_records)
official_test_df.to_csv(OFFICIAL_TEST_CSV, index=False)

test_clip_set = set(official_test_df['rel_path'])

print(f"Official test set: {len(test_clip_set):,} clips")
print("\nBreakdown by folder:")
folder_counts = Counter(p.split('/')[0] for p in test_clip_set)
for folder, count in sorted(folder_counts.items()):
    print(f"  {folder}: {count}")

print("\nBreakdown by official label (1=real, 0=fake):")
label_counts = official_test_df['official_label'].value_counts().to_dict()
for label, count in sorted(label_counts.items()):
    descriptor = 'REAL' if label == '1' else 'FAKE'
    print(f"  {label} ({descriptor}): {count}")

print("\nFirst 5 entries:")
for _, row in official_test_df.head().iterrows():
    print(f"  {row['official_label']} {row['rel_path']}")

## 5. Filename parser

**Key change from v1:** for `Celeb-synthesis` clips, `actor_ids` now contains ONLY the source identity (idN, the face donor). The target identity (idM, body/audio donor) is recorded separately but NOT used for split decisions. This is the protocol used by Celeb-DF papers and is what avoids the v1 collapse.

In [ ]:
def parse_filename(rel_path: str) -> dict | None:
    parts = rel_path.replace('\\', '/').split('/')
    if len(parts) < 2:
        return None
    folder = parts[0]
    filename = parts[-1]
    if not filename.endswith('.mp4'):
        return None
    name = filename[:-4]
    
    if folder == 'Celeb-real':
        m = re.match(r'^(id\d+)_(\d+)$', name)
        if not m: return None
        sid, clip_idx = m.group(1), m.group(2)
        return {
            'folder': folder, 'filename': filename, 'rel_path': rel_path,
            'label': 'REAL', 'source_id': sid, 'target_id': None,
            'clip_idx': clip_idx, 'actor_ids': frozenset([sid]),
        }
    elif folder == 'Celeb-synthesis':
        m = re.match(r'^(id\d+)_(id\d+)_(\d+)$', name)
        if not m: return None
        sid, tid, clip_idx = m.group(1), m.group(2), m.group(3)
        # ACTOR_IDS = source only — target can span splits
        return {
            'folder': folder, 'filename': filename, 'rel_path': rel_path,
            'label': 'FAKE', 'source_id': sid, 'target_id': tid,
            'clip_idx': clip_idx, 'actor_ids': frozenset([sid]),
        }
    elif folder == 'YouTube-real':
        m = re.match(r'^(\d+)$', name)
        if not m: return None
        clip_idx = m.group(1)
        actor = f'yt_{clip_idx}'
        return {
            'folder': folder, 'filename': filename, 'rel_path': rel_path,
            'label': 'REAL', 'source_id': actor, 'target_id': None,
            'clip_idx': clip_idx, 'actor_ids': frozenset([actor]),
        }
    return None


# Sanity check
for t in ['Celeb-real/id0_0000.mp4', 'Celeb-synthesis/id0_id16_0042.mp4',
          'YouTube-real/00050.mp4', 'Celeb-real/garbage.mp4']:
    print(f"  {t} -> {parse_filename(t)}")

## 6. Train/Val hashing

Each identity hashes to either 'train' (90%) or 'val' (10%). Test is excluded from this — it's pre-assigned from the official list.

In [ ]:
def actor_to_train_val(actor_id: str, ratios: dict = TRAIN_VAL_RATIOS, seed: int = SAMPLE_SEED) -> str:
    h = hashlib.md5(f"{seed}_{actor_id}".encode()).hexdigest()
    n = int(h, 16) / (16 ** 32)
    cum = 0.0
    for split, ratio in ratios.items():
        cum += ratio
        if n < cum:
            return split
    return list(ratios.keys())[-1]


# Sanity check: hash distribution on 500 synthetic identities
test_ids = [f'id{i}' for i in range(60)] + [f'yt_{i:05d}' for i in range(300)]
test_splits = [actor_to_train_val(a) for a in test_ids]
test_counts = Counter(test_splits)
print("Hash distribution on synthetic IDs:")
for split in ['train', 'val']:
    pct = test_counts[split] / len(test_ids) * 100
    expected = TRAIN_VAL_RATIOS[split] * 100
    print(f"  {split}: {test_counts[split]:3d} ({pct:5.1f}% — expected {expected:.0f}%)")
assert actor_to_train_val('id0') == actor_to_train_val('id0'), "Hash not deterministic"
print("\n[OK] Hashing deterministic.")

## 7. Build full manifest

For each `.mp4` in `data/celeb-df/`:
1. Parse the filename → REAL/FAKE, source_id, target_id
2. If the clip is in the official test list → split = 'test'
3. Otherwise → hash `source_id` to 'train' or 'val'

In [ ]:
rows = []
parse_failures = []

for subfolder in ['Celeb-real', 'Celeb-synthesis', 'YouTube-real']:
    folder_path = CELEB_DIR / subfolder
    if not folder_path.exists():
        print(f"[WARN] Folder missing: {folder_path}")
        continue
    mp4_files = sorted(folder_path.glob('*.mp4'))
    print(f"{subfolder}: {len(mp4_files):,} videos")
    
    for f in mp4_files:
        rel_path = f"{subfolder}/{f.name}"
        meta = parse_filename(rel_path)
        if meta is None:
            parse_failures.append(rel_path)
            continue
        
        # Assign split
        if rel_path in test_clip_set:
            meta['split'] = 'test'
        else:
            # Hash by source identity
            sid = meta['source_id']
            meta['split'] = actor_to_train_val(sid)
        
        meta['actor_ids_str'] = '|'.join(sorted(meta['actor_ids']))
        rows.append(meta)

print(f"\nTotal manifest rows: {len(rows):,}")
print(f"Parse failures: {len(parse_failures)}")
if parse_failures:
    print(f"  Examples: {parse_failures[:5]}")

## 8. Save manifests

In [ ]:
cols = ['rel_path', 'folder', 'filename', 'label', 'source_id', 'target_id',
        'clip_idx', 'actor_ids_str', 'split']

manifest_df = pd.DataFrame(rows)[cols]
manifest_df.to_csv(MANIFEST_CSV, index=False)
print(f"Wrote {MANIFEST_CSV}: {len(manifest_df):,} rows")

for split in ['train', 'val', 'test']:
    split_df = manifest_df[manifest_df['split'] == split].copy()
    out_path = SPLITS_DIR / f'{split}.csv'
    split_df.to_csv(out_path, index=False)
    print(f"  {split}: {len(split_df):,} clips → {out_path.name}")

## 9. Verification — split sizes, balance, leakage check

In [ ]:
manifest_df = pd.read_csv(MANIFEST_CSV)

print("=" * 60)
print("MANIFEST OVERVIEW")
print("=" * 60)
print(f"Total clips:              {len(manifest_df):,}")
print(f"Unique source identities: {manifest_df['source_id'].nunique():,}")

print("\n--- Split sizes by label ---")
split_label = manifest_df.groupby(['split', 'label']).size().unstack(fill_value=0)
split_label['TOTAL'] = split_label.sum(axis=1)
print(split_label)

print("\n--- Split sizes by source folder ---")
split_folder = manifest_df.groupby(['split', 'folder']).size().unstack(fill_value=0)
print(split_folder)

print("\n--- Class balance per split ---")
for split in ['train', 'val', 'test']:
    sub = manifest_df[manifest_df['split'] == split]
    if len(sub) == 0:
        continue
    n_real = (sub['label'] == 'REAL').sum()
    n_fake = (sub['label'] == 'FAKE').sum()
    print(f"  {split}: REAL={n_real:,} ({n_real/len(sub)*100:.1f}%), FAKE={n_fake:,} ({n_fake/len(sub)*100:.1f}%)")

# --- Leakage check: source identities must not appear in multiple non-test splits ---
print("\n--- Source-identity leakage check (train vs val) ---")
non_test = manifest_df[manifest_df['split'].isin(['train', 'val'])]
source_splits = non_test.groupby('source_id')['split'].nunique()
leaked = source_splits[source_splits > 1]
if len(leaked) == 0:
    n_unique = source_splits.shape[0]
    print(f"  [OK] All {n_unique:,} non-test source identities appear in exactly one split (train OR val).")
else:
    print(f"  [FAIL] {len(leaked)} source identities appear in both train and val!")
    print(leaked.head(10))
    raise AssertionError("Source-identity leakage between train and val")

# Test set leakage check: source identities of test clips should not appear in train/val
# This is a separate check because test is defined by Celeb-DF (not by our hash) — so it's a sanity check
test_sources = set(manifest_df[manifest_df['split']=='test']['source_id'])
train_val_sources = set(non_test['source_id'])
overlap = test_sources & train_val_sources
if overlap:
    print(f"\n  [NOTE] {len(overlap)} source identities appear in BOTH test and train/val:")
    print(f"  This means Celeb-DF's official test set includes identities also present in train.")
    print(f"  Example overlapping identities: {sorted(overlap)[:10]}")
    print(f"  This is *expected* for the official split and is what every Celeb-DF paper uses.")
    print(f"  Our train-val split remains strictly actor-disjoint among non-test clips.")
else:
    print(f"\n  [INFO] Test source identities are entirely disjoint from train/val sources.")

## 10. Disk usage summary

In [ ]:
def folder_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / (1024**3)

print("Disk usage:")
print(f"  data/raw_zips/     : {folder_size_gb(DATA_DIR / 'raw_zips'):.2f} GB")
print(f"  data/celeb-df/     : {folder_size_gb(CELEB_DIR):.2f} GB")
print(f"  data/splits/       : {folder_size_gb(SPLITS_DIR):.4f} GB")
print(f"\n  Free disk:         : {get_free_gb(DATA_DIR):.1f} GB")

## 11. Next steps

**Expected ballpark numbers** (your output should match closely):
- Total clips: ~6,500
- Test: ~518 clips (Celeb-DF's official test set)
- Train: ~5,400 clips (~90% of remaining)
- Val: ~600 clips (~10% of remaining)
- All splits should have a meaningful number of both REAL and FAKE clips (no extreme imbalance per split)

If everything looks good:
1. **Update `project_log.md`** with the final stats — paste the verification output
2. **Commit + push** the updated notebook
3. **Move on to notebook 02** (`02_preprocessing.ipynb`) — MTCNN face detection + 16kHz audio extraction